# ChaoSSM — Phase 2a: Associative-Recall Mechanism (RSA)

## Why this phase

Phase 1 found linear beats chaossm_free on associative recall, cleanly (5/5 seeds, paired
Wilcoxon p=0.0625). The stated hypothesis was that chaotic sensitivity corrupts precise
key-value addressing — but that was an interpretation of a number, not a tested mechanism.
This phase tests it directly.

The Phase 1d precedent for what a good mechanistic follow-up looks like: selective-copy's
variance turned out to be explained almost exactly by which side of the S5.2 plateau boundary
each seed's λ settled on (mean accuracy 0.129 inside vs. 0.071 outside — split precisely at
λ=0.4). This phase looks for the equivalent explanation for recall.

## The task's structure, and what "addressing" means concretely

Associative recall presents M (key, value) pairs, then a query key, and the model must output
the paired value. For a linear readout at the final position to succeed, the hidden state at
query time must end up representationally *close* to wherever the matching key's information
was stored, and *far* from the other M-1 stored pairs — otherwise the readout has no way to
select the right one. This gives two directly measurable, non-circular quantities (computed on
the hidden-state trajectory, not the trained readout itself, so this doesn't just re-measure
task accuracy):

- **Matching margin**: cosine similarity between the query-time state and the state
  immediately after the *matching* pair was presented, minus the mean similarity to the
  *non-matching* pairs' states. A working addressing mechanism should show a positive,
  reliable margin. Chaotic scrambling would show up as a smaller or noisier margin.
- **Inter-key separability**: mean pairwise cosine similarity *among* the M stored-pair
  states within a sequence. Lower (more orthogonal) means the M stored items are easier to
  tell apart in principle, independent of whether the query correctly finds the right one.
  Higher means the stored representations are collapsing toward each other — a distinct
  failure mode from a bad margin.

Together these separate two different ways recall could fail: the model confuses stored items
with each other (poor separability), or it stores them fine but the query doesn't align with
the right one (poor margin despite good separability). Both are computed via forward passes
on a trained, frozen model — no additional training, no risk of the analysis just re-deriving
task accuracy through a different route.

**Caveat stated up front**: `linear`'s cell is tanh-bounded ([-1,1]) and the chaotic arms'
cell is fold-bounded ([0,1]); raw cosine-similarity *levels* are not directly comparable
across arms for this reason. The margin and separability *contrasts* (matching vs.
non-matching, within-model) are the comparable quantities, and are what this phase reports.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase2a"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase2a


## Configuration

Task and architecture identical to Phase 1's associative-recall setting (`recall_pairs=6`,
`steps=1200`). `seeds=10` — more than Phase 1's 5 for better-powered correlation analysis,
less than Phase 1d's 15 since recall already showed a clean 5/5-seed effect and doesn't need
as much power just to detect a difference. `rsa_eval_batch=512` fresh, untrained-on sequences
per model for the representational analysis.

In [2]:
SMOKE_TEST = False   # full run: 3 arms x 10 seeds, associative recall + RSA

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    recall_pairs: int = 6
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    seeds: int = 10
    eval_every: int = 20
    rsa_eval_batch: int = 512

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2; CFG.rsa_eval_batch = 64
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "recall_pairs": "6",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "seeds": "10",
  "eval_every": "20",
  "rsa_eval_batch": "512"
}


## 1. Model — unchanged from Phase 1

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimator — unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 3. Associative-recall task, with query index exposed for RSA

Identical to Phase 1's `make_assoc_recall_batch`, with `qidx` (which of the M pairs was
queried) additionally returned — needed to identify the matching vs. non-matching stored
states during analysis. Every generator call carries `device=device` explicitly, matching the
already-audited Phase 1 version this is copied from.

In [5]:
def make_assoc_recall_batch(cfg, batch, device, gen):
    M, V = cfg.recall_pairs, cfg.vocab
    sep = V
    keys = torch.stack([torch.randperm(V, generator=gen, device=device)[:M] for _ in range(batch)])
    vals = torch.randint(0, V, (batch, M), device=device, generator=gen)
    qidx = torch.randint(0, M, (batch,), device=device, generator=gen)
    bidx = torch.arange(batch, device=device)
    qkey = keys[bidx, qidx]
    qval = vals[bidx, qidx]
    pairs = torch.stack([keys, vals], dim=2).reshape(batch, 2*M)   # k1,v1,k2,v2,...
    inp = torch.cat([pairs, torch.full((batch,1), sep, device=device), qkey.unsqueeze(1)], 1)
    tgt = torch.full((batch, inp.shape[1]), -100, device=device)
    tgt[:, -1] = qval
    return inp, tgt, qidx

VOCAB_SIZE, READOUT_SIZE = CFG.vocab + 1, CFG.vocab + 1
CHANCE = 1.0 / CFG.vocab
# pair i (1-indexed) ends at sequence position 2i-1 (0-indexed): after k_i,v_i
PAIR_END_POS = [2*i - 1 for i in range(1, CFG.recall_pairs + 1)]
print("pair-end positions:", PAIR_END_POS, "| query position: -1 (last)")

pair-end positions: [1, 3, 5, 7, 9, 11] | query position: -1 (last)


## 4. Trainer — unchanged mechanics

In [6]:
def train_one(arm, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, READOUT_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt, _ = make_assoc_recall_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.eval_every == 0:
            with torch.no_grad():
                acc = (logits[:, -1].argmax(-1) == tgt[:, -1]).float().mean().item()
            hist.append(dict(step=step, arm=arm, seed=seed, loss=L_task.item(), acc=acc,
                             lam=lam_val, gnorm=gn))

    tail = hist[-max(1, len(hist)//5):]
    summary = dict(arm=arm, seed=seed, diverged=diverged,
                   final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
                   lam_settled=float(np.nanmean([h["lam"] for h in tail])) if tail else np.nan)
    return model, summary, pd.DataFrame(hist)

## 5. Train all (arm, seed) models

In [7]:
models = {}
rows, hists = [], []
total = len(CFG.arms) * CFG.seeds
done = 0
for arm in CFG.arms:
    for sd in range(CFG.seeds):
        model, s, h = train_one(arm, sd, CFG)
        models[(arm, sd)] = model
        rows.append(s); hists.append(h); done += 1
        flag = " DIVERGED" if s["diverged"] else ""
        print(f"[{done}/{total}] {arm:18s} seed={sd:2d}  final_acc={s['final_acc']:.4f} "
              f"(chance {CHANCE:.4f})  lam={s['lam_settled']:+.3f}{flag}")

TRAIN_SUM = pd.DataFrame(rows); TRAIN_HIST = pd.concat(hists, ignore_index=True)
savetable(TRAIN_SUM, "phase2a_train_summary.csv")

[1/30] linear             seed= 0  final_acc=0.1589 (chance 0.0625)  lam=+nan
[2/30] linear             seed= 1  final_acc=0.1615 (chance 0.0625)  lam=+nan
[3/30] linear             seed= 2  final_acc=0.1471 (chance 0.0625)  lam=+nan
[4/30] linear             seed= 3  final_acc=0.1510 (chance 0.0625)  lam=+nan
[5/30] linear             seed= 4  final_acc=0.1549 (chance 0.0625)  lam=+nan
[6/30] linear             seed= 5  final_acc=0.1615 (chance 0.0625)  lam=+nan
[7/30] linear             seed= 6  final_acc=0.1393 (chance 0.0625)  lam=+nan
[8/30] linear             seed= 7  final_acc=0.1536 (chance 0.0625)  lam=+nan
[9/30] linear             seed= 8  final_acc=0.1341 (chance 0.0625)  lam=+nan
[10/30] linear             seed= 9  final_acc=0.1641 (chance 0.0625)  lam=+nan
[11/30] chaossm_free       seed= 0  final_acc=0.0859 (chance 0.0625)  lam=+0.354
[12/30] chaossm_free       seed= 1  final_acc=0.1003 (chance 0.0625)  lam=+0.196
[13/30] chaossm_free       seed= 2  final_acc=0.0729 (cha

## 6. Representational similarity analysis

For each trained model: run `rsa_eval_batch` fresh sequences (no gradient, no training),
capture the full state trajectory, and compute the matching margin and inter-key
separability defined in the introduction.

In [8]:
@torch.no_grad()
def rsa_analysis(model, arm, cfg, n_eval, seed_offset=10_000):
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu")
    gen.manual_seed(seed_offset)  # fixed, held-out eval stream, distinct from any training seed
    inp, tgt, qidx = make_assoc_recall_batch(cfg, n_eval, DEVICE, gen)
    need_traj = (arm != "linear")
    if not need_traj:
        # still need trajectories for RSA even for the linear arm
        out = model(inp, return_traj=True)
        logits, traj, _ = out
    else:
        logits, traj, _ = model(inp, return_traj=True)

    h_query = traj[:, -1, :]                                    # (B, N)
    h_pairs = torch.stack([traj[:, p, :] for p in PAIR_END_POS], dim=1)   # (B, M, N)

    hq = F.normalize(h_query, dim=-1)
    hp = F.normalize(h_pairs, dim=-1)
    sims = torch.einsum("bn,bmn->bm", hq, hp)                    # (B, M) cos(h_query, h_pair_i)

    bidx = torch.arange(n_eval, device=DEVICE)
    matching_sim = sims[bidx, qidx]                              # (B,)
    mask = torch.ones_like(sims, dtype=torch.bool)
    mask[bidx, qidx] = False
    nonmatching_sim = sims[mask].reshape(n_eval, cfg.recall_pairs - 1).mean(dim=1)
    margin = (matching_sim - nonmatching_sim)                    # (B,)

    # inter-key separability: mean pairwise cos sim among the M stored states, per sequence
    sim_matrix = torch.einsum("bmn,bkn->bmk", hp, hp)            # (B, M, M)
    eye = torch.eye(cfg.recall_pairs, device=DEVICE, dtype=torch.bool).unsqueeze(0)
    offdiag = sim_matrix.masked_select(~eye.expand(n_eval, -1, -1)).reshape(n_eval, -1)
    separability = offdiag.mean(dim=1)                           # (B,) -- higher = more collapsed

    acc = (logits[:, -1].argmax(-1) == tgt[:, -1]).float()

    return dict(margin_mean=float(margin.mean()), margin_std=float(margin.std()),
                separability_mean=float(separability.mean()), separability_std=float(separability.std()),
                eval_acc=float(acc.mean()))

rsa_rows = []
for (arm, sd), model in models.items():
    r = rsa_analysis(model, arm, CFG, CFG.rsa_eval_batch)
    r["arm"] = arm; r["seed"] = sd
    r["lam_settled"] = TRAIN_SUM[(TRAIN_SUM.arm==arm)&(TRAIN_SUM.seed==sd)].lam_settled.iloc[0]
    rsa_rows.append(r)
    print(f"{arm:18s} seed={sd:2d}  margin={r['margin_mean']:+.4f}  "
          f"separability={r['separability_mean']:+.4f}  eval_acc={r['eval_acc']:.4f}")

RSA = pd.DataFrame(rsa_rows)
savetable(RSA, "phase2a_rsa.csv")

linear             seed= 0  margin=+0.1928  separability=+0.1965  eval_acc=0.1230
linear             seed= 1  margin=+0.1741  separability=+0.2468  eval_acc=0.1641
linear             seed= 2  margin=+0.1889  separability=+0.2116  eval_acc=0.1582
linear             seed= 3  margin=+0.2013  separability=+0.2044  eval_acc=0.1426
linear             seed= 4  margin=+0.1984  separability=+0.1828  eval_acc=0.1562
linear             seed= 5  margin=+0.1920  separability=+0.2240  eval_acc=0.1426
linear             seed= 6  margin=+0.1934  separability=+0.2002  eval_acc=0.1523
linear             seed= 7  margin=+0.1794  separability=+0.1724  eval_acc=0.1230
linear             seed= 8  margin=+0.1963  separability=+0.2156  eval_acc=0.1602
linear             seed= 9  margin=+0.2133  separability=+0.1961  eval_acc=0.1367
chaossm_free       seed= 0  margin=+0.0045  separability=+0.7677  eval_acc=0.0938
chaossm_free       seed= 1  margin=-0.0020  separability=+0.7820  eval_acc=0.1211
chaossm_free    

## 7. Does margin / separability explain the accuracy gap?

In [9]:
print(RSA.groupby("arm")[["margin_mean","separability_mean","eval_acc"]].agg(["mean","std"]).round(4).to_string())

def paired(a, b, col):
    x = RSA[RSA.arm==a].sort_values("seed")[col].values
    y = RSA[RSA.arm==b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(pair=f"{a} vs {b}", metric=col, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(pair=f"{a} vs {b}", metric=col, n=len(x), p=float(p), median_diff=float(np.median(x-y)))

rows = []
for metric in ["margin_mean", "separability_mean"]:
    rows.append(paired("linear", "chaossm_free", metric))
    rows.append(paired("linear", "chaossm_corridor", metric))
    rows.append(paired("chaossm_corridor", "chaossm_free", metric))
ST = pd.DataFrame(rows)
print()
print(ST.to_string(index=False))
savetable(ST, "phase2a_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

                 margin_mean         separability_mean         eval_acc        
                        mean     std              mean     std     mean     std
arm                                                                            
chaossm_corridor     -0.0039  0.0032            0.7817  0.0124   0.0947  0.0149
chaossm_free         -0.0024  0.0036            0.7814  0.0108   0.0992  0.0186
linear                0.1930  0.0110            0.2051  0.0211   0.1459  0.0148

                            pair            metric  n        p  median_diff
          linear vs chaossm_free       margin_mean 10 0.001953     0.194388
      linear vs chaossm_corridor       margin_mean 10 0.001953     0.197483
chaossm_corridor vs chaossm_free       margin_mean 10 0.130859    -0.001924
          linear vs chaossm_free separability_mean 10 0.001953    -0.580846
      linear vs chaossm_corridor separability_mean 10 0.001953    -0.578879
chaossm_corridor vs chaossm_free separability_mean 10 0.921875 

In [10]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
colors = {"linear":"gray", "chaossm_free":"tab:blue", "chaossm_corridor":"tab:green"}
for arm in CFG.arms:
    d = RSA[RSA.arm==arm]
    ax[0].scatter(d.margin_mean, d.eval_acc, label=arm, color=colors[arm], alpha=.8)
ax[0].set(xlabel="matching margin", ylabel="recall accuracy",
          title="Does margin predict accuracy?")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

RSA.boxplot(column="margin_mean", by="arm", ax=ax[1])
ax[1].set(title="Matching margin by arm", xlabel="arm", ylabel="margin")

RSA.boxplot(column="separability_mean", by="arm", ax=ax[2])
ax[2].set(title="Inter-key separability by arm\n(higher = more collapsed)", xlabel="arm", ylabel="separability")
fig.suptitle(""); fig.tight_layout(); savefig(fig, "phase2a_margin_separability.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase2a/phase2a_margin_separability.png


## 8. Does λ predict the mechanism, within the chaotic arms?

The Phase 1d precedent: seeds landing inside the S5.2 plateau [0, 0.4] averaged much higher
selective-copy accuracy than seeds that drifted past it. If the same mechanism explains recall,
higher λ should correlate with a *smaller* margin and/or *higher* (more collapsed)
separability within `chaossm_free`.

In [11]:
free = RSA[RSA.arm == "chaossm_free"]
corr_margin = free[["lam_settled","margin_mean"]].corr().iloc[0,1]
corr_sep = free[["lam_settled","separability_mean"]].corr().iloc[0,1]
print(f"corr(lambda, margin) within chaossm_free:         {corr_margin:+.3f}")
print(f"corr(lambda, separability) within chaossm_free:   {corr_sep:+.3f}")

plateau_mask = free.lam_settled <= 0.4
n_in, n_out = int(plateau_mask.sum()), int((~plateau_mask).sum())
m_in = free[plateau_mask].margin_mean.mean() if n_in else float("nan")
m_out = free[~plateau_mask].margin_mean.mean() if n_out else float("nan")
a_in = free[plateau_mask].eval_acc.mean() if n_in else float("nan")
a_out = free[~plateau_mask].eval_acc.mean() if n_out else float("nan")
print(f"\nmean margin  | lambda<=0.4 (n={n_in}): {m_in:+.4f}  | lambda>0.4 (n={n_out}): {m_out:+.4f}")
print(f"mean acc     | lambda<=0.4 (n={n_in}): {a_in:.4f}  | lambda>0.4 (n={n_out}): {a_out:.4f}")
if n_in == 0 or n_out == 0:
    print("(note: one bucket is empty at this seed count -- the split is not yet informative)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(free.lam_settled, free.margin_mean, alpha=.8)
ax[0].axvline(0.4, color="green", ls="--", label="S5.2 plateau boundary")
ax[0].set(xlabel=r"settled $\lambda$", ylabel="matching margin", title=f"corr={corr_margin:+.3f}")
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].scatter(free.lam_settled, free.separability_mean, alpha=.8, color="tab:orange")
ax[1].axvline(0.4, color="green", ls="--", label="S5.2 plateau boundary")
ax[1].set(xlabel=r"settled $\lambda$", ylabel="inter-key separability", title=f"corr={corr_sep:+.3f}")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase2a_lambda_mechanism.png"); plt.close(fig)

corr(lambda, margin) within chaossm_free:         +0.185
corr(lambda, separability) within chaossm_free:   +0.486

mean margin  | lambda<=0.4 (n=8): -0.0021  | lambda>0.4 (n=2): -0.0036
mean acc     | lambda<=0.4 (n=8): 0.1047  | lambda>0.4 (n=2): 0.0771
saved /content/drive/My Drive/ChaoSSM/results/phase2a/phase2a_lambda_mechanism.png


## 9. Verdict

| Observation | Reading |
|---|---|
| linear shows significantly larger margin than chaossm_free, separability similar | Confirms the addressing-precision hypothesis directly: chaos's disadvantage is in query-to-key alignment, not general item confusability. This is the clean, publishable mechanistic result. |
| linear shows lower (better) separability than chaossm_free, margin similar | The failure mode is item confusability (stored representations collapsing toward each other), not query alignment — a different, still-interesting mechanism than the one hypothesized. |
| Both margin and separability differ significantly, same direction | Chaos hurts recall on both axes simultaneously — the strongest form of the scrambling hypothesis. |
| Neither margin nor separability differs significantly despite the accuracy gap | The accuracy gap is real but this pair of metrics doesn't explain it — the mechanism is elsewhere (e.g. in how the readout weights interact with the representation, not in the representational geometry itself). Report as a partial answer, not a null result on the original accuracy finding. |
| corr(λ, margin) or corr(λ, separability) within chaossm_free replicates the S5.2/S5.4 plateau story | Ties the recall mechanism directly to the same λ-plateau explanation already established for selective copying — a strong, unifying result across two independent tasks. |
| No λ correlation within chaossm_free | The recall degradation is a property of chaos generally, not specifically of operating past the S5.2 plateau — a real distinction from the selective-copy story worth stating precisely. |

In [12]:
rsa_summary_df = RSA.groupby("arm")[["margin_mean","separability_mean","eval_acc"]].agg(["mean","std"]).round(4)
rsa_summary_df.columns = ["_".join(c) for c in rsa_summary_df.columns]   # flatten MultiIndex -> plain strings

verdict = dict(
    smoke_test=SMOKE_TEST,
    train_summary=TRAIN_SUM.groupby("arm").final_acc.agg(["mean","std"]).round(4).to_dict("index"),
    rsa_summary=rsa_summary_df.to_dict("index"),
    stats=ST.to_dict("records"),
    corr_lambda_margin_chaossm_free=float(corr_margin),
    corr_lambda_separability_chaossm_free=float(corr_sep),
    plateau_split=dict(
        n_in_plateau=n_in, n_above_plateau=n_out,
        margin_in_plateau=float(m_in) if n_in else None,
        margin_above_plateau=float(m_out) if n_out else None,
    ),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, recall_pairs=CFG.recall_pairs,
                rsa_eval_batch=CFG.rsa_eval_batch),
)
savejson(verdict, "phase2a_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase2a/phase2a_verdict.json
{
  "smoke_test": false,
  "train_summary": {
    "chaossm_corridor": {
      "mean": 0.0901,
      "std": 0.0154
    },
    "chaossm_free": {
      "mean": 0.0927,
      "std": 0.0134
    },
    "linear": {
      "mean": 0.1526,
      "std": 0.0099
    }
  },
  "rsa_summary": {
    "chaossm_corridor": {
      "margin_mean_mean": -0.0039,
      "margin_mean_std": 0.0032,
      "separability_mean_mean": 0.7817,
      "separability_mean_std": 0.0124,
      "eval_acc_mean": 0.0947,
      "eval_acc_std": 0.0149
    },
    "chaossm_free": {
      "margin_mean_mean": -0.0024,
      "margin_mean_std": 0.0036,
      "separability_mean_mean": 0.7814,
      "separability_mean_std": 0.0108,
      "eval_acc_mean": 0.0992,
      "eval_acc_std": 0.0186
    },
    "linear": {
      "margin_mean_mean": 0.193,
      "margin_mean_std": 0.011,
      "separability_mean_mean": 0.2051,
      "separability_mean_std": 0.0211,
      "ev